### 解析GSE.soft文件

In [1]:
import re
import pandas as pd
import os

def extract_information(file_path):
    """从GSE的soft文件中提取提交单位与国家
    """

    information = []

    with open(file_path) as file:
        content = file.read()

    # 使用正则表达式进行匹配提取信息
    accession_pattern = r'!Series_geo_accession = (\w+)'
    accessions = re.findall(accession_pattern, content)
    information.append(accessions[0])

    institute_pattern = r'!Series_contact_institute = (.*)'
    institutes = re.findall(institute_pattern, content)
    information.append(institutes[0])

    country_pattern = r'!Series_contact_country = (.*)'
    countries = re.findall(country_pattern, content)
    information.append(countries[0])

    organism_pattern = r'!Series_platform_organism = (.*)'
    organism = re.findall(organism_pattern, content)
    information.append(organism[0])

    return information

In [2]:
"""
单个测试
"""
location = './Mt/GSE_soft/GSE110062_GSE.soft'
extract_information(location)

['GSE110062',
 'Tianjin institute of industrial biotechnology,Chinses Academy of Sciences',
 'China',
 'Thermothelomyces thermophilus']

In [3]:
"""
批量处理
"""

path = './Mt/GSE_soft/'
key_name = 'Mt'
results = []

for filename in os.listdir(path):
    result = extract_information(os.path.join(path, filename))
    results.append(result)

df = pd.DataFrame(results, columns=['GEO Series ID', 'Institute', 'Country', 'Organism'])
display(df)
# df.to_csv(key_name + '_soft.csv', index=False )
df.to_excel(key_name + '_soft.xlsx', index=False)


,GEO Series ID,Institute,Country,Organism
0,GSE111986,"Tianjin institute of industrial biotechnology,...",China,Neurospora crassa
1,GSE214142,Chinese Academy of Sciences,China,Thermothelomyces thermophilus ATCC 42464
2,GSE110062,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus
3,GSE183387,Technische Universität Berlin,Germany,Thermothelomyces thermophilus
4,GSE165516,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus
5,GSE57235,Concordia University,Canada,Thermothelomyces thermophilus
6,GSE184074,"Tianjin Institute of Industrial Biotechnology,...",China,Thermothelomyces thermophilus ATCC 42464
7,GSE137286,Centre of fungal biodiversity,Netherlands,Thermothelomyces thermophilus
8,GSE205182,TIB CAS,China,Thermothelomyces thermophilus
9,GSE213570,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus ATCC 42464


In [4]:
"""
整合Mt的数据集信息
"""

# 读取Mt_gse_info.xlsx
df1 = pd.read_excel('Mt_gse_info.xlsx')

# df1取GEO Series ID列，Samples列，PMID列
df1 = df1[['GSE Series', 'Samples', 'PMID']]

# PMID列处理为整数
df1['PMID'] = df1['PMID'].fillna(0)
df1['PMID'] = df1['PMID'].astype(int)

# PMID列有重复，相同的PMID对应的只保留最后一行
# df1 = df1.drop_duplicates(subset=['PMID'], keep='last')
df1

,GSE Series,Samples,PMID
0,GSE213576,36,37013645
1,GSE213575,4,37013645
2,GSE213573,8,37013645
3,GSE213570,24,37013645
4,GSE221464,24,36966330
5,GSE214002,8,36691040
6,GSE214142,18,36478865
7,GSE205182,18,36165620
8,GSE184074,12,35257374
9,GSE183387,49,0


In [5]:
df2 = pd.read_excel('Mt_gse_info_keywords.xlsx')

# PMID列处理为整数
df2['PMID'] = df2['PMID'].astype(int)

# PMID列有重复，相同的PMID对应的只保留最后一行
df2 = df2.drop_duplicates(subset=['PMID'], keep='last')

df2

,PMID,Title,Abstract,Keywords,Journal,Country,Year
3,37013645,The putative methyltransferase LaeA regulates ...,BACKGROUND: Filamentous fungi with the ability...,"['Cellulase', 'Fungal growth', 'Gene regulatio...",Biotechnology for biofuels and bioproducts,China,2023 Apr 3
4,36966330,The arabinose transporter MtLat-1 is involved ...,BACKGROUND: Filamentous fungi possess an array...,"['Hemicellulase', 'L-Arabinose transport', 'Mt...",Biotechnology for biofuels and bioproducts,China,2023 Mar 25
5,36691040,The Weimberg pathway: an alternative for Mycel...,BACKGROUND: With D-xylose being the second mos...,"['1,2,4-Butanetriol', 'D-Xylose', 'D-Xylose de...",Biotechnology for biofuels and bioproducts,China,2023 Jan 23
6,36478865,PFK2/FBPase-2 is a potential target for metabo...,The key enzyme 6-phosphofructo-2-kinase (PFK2)...,"['6-phosphofructo-2-kinase/fructose-2,6-biphos...",Frontiers in microbiology,China,2022
7,36165620,"MtTRC-1, a Novel Transcription Factor, Regulat...",The thermophilic fungus Myceliophthora thermop...,"['ER stress', 'MtHAC-1', 'MtTRC-1', 'Mtcbh-1',...",Applied and environmental microbiology,China,2022 Oct 11
8,35257374,Reconstruction and analysis of genome-scale me...,"Myceliophthora thermophila, a thermophilic fun...","['Myceliophthora thermophila', 'genome-scale m...",Biotechnology and bioengineering,China,2022 Jul
9,31078793,Direct production of commodity chemicals from ...,The production of fuels and chemicals from ren...,"['Biochemicals', 'C4-diacid', 'CRISPR/Cas9', '...",Metabolic engineering,China,2020 Sep
10,30474279,"CLR-4, a novel conserved transcription factor ...",Fungal degradation of lignocellulosic biomass ...,['?'],Molecular microbiology,China,2019 Feb
11,33995328,Transcriptional Profiling of Myceliophthora th...,Efficient biological conversion of all sugars ...,"['Myceliophthora', 'galactose transport', 'gal...",Frontiers in microbiology,China,2021
12,29843921,Transcriptional analysis of Myceliophthora the...,Thermophilic fungus Myceliophthora thermophila...,"['Cellulose', 'Myceliophthora thermophila', 'S...",Bioresource technology,China,2018 Oct


In [6]:
df3 = pd.read_excel('Mt_soft.xlsx')

# GEO Series ID列更名为GSE Series
df3 = df3.rename(columns={'GEO Series ID': 'GSE Series'})

df3

,GSE Series,Institute,Country,Organism
0,GSE111986,"Tianjin institute of industrial biotechnology,...",China,Neurospora crassa
1,GSE214142,Chinese Academy of Sciences,China,Thermothelomyces thermophilus ATCC 42464
2,GSE110062,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus
3,GSE183387,Technische Universität Berlin,Germany,Thermothelomyces thermophilus
4,GSE165516,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus
5,GSE57235,Concordia University,Canada,Thermothelomyces thermophilus
6,GSE184074,"Tianjin Institute of Industrial Biotechnology,...",China,Thermothelomyces thermophilus ATCC 42464
7,GSE137286,Centre of fungal biodiversity,Netherlands,Thermothelomyces thermophilus
8,GSE205182,TIB CAS,China,Thermothelomyces thermophilus
9,GSE213570,"Tianjin institute of industrial biotechnology,...",China,Thermothelomyces thermophilus ATCC 42464


In [7]:
# df1与df2合并，依据PMID列, 以df1为主
df4 = pd.merge(df1, df2, on='PMID', how='left')

# df去掉Country列
df4 = df4.drop(columns=['Country'])

# df4中的PMID中的0值替换为空
df4['PMID'] = df4['PMID'].replace(0, '')

# df4中每一列的空值替换为空
df4 = df4.fillna('')

# Keywords列中的中括号替换为空
df4['Keywords'] = df4['Keywords'].str.replace('[', '')
df4['Keywords'] = df4['Keywords'].str.replace(']', '')

# df4中的Abstract列删除BACKGROUND:
df4['Abstract'] = df4['Abstract'].str.replace('BACKGROUND:', '')

# df4中的Yead列显示的是年份+月份+日期，只保留年份与月份
df4['Year'] = df4['Year'].str[:8]

df4

,GSE Series,Samples,PMID,Title,Abstract,Keywords,Journal,Year
0,GSE213576,36,37013645,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,"'Cellulase', 'Fungal growth', 'Gene regulation...",Biotechnology for biofuels and bioproducts,2023 Apr
1,GSE213575,4,37013645,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,"'Cellulase', 'Fungal growth', 'Gene regulation...",Biotechnology for biofuels and bioproducts,2023 Apr
2,GSE213573,8,37013645,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,"'Cellulase', 'Fungal growth', 'Gene regulation...",Biotechnology for biofuels and bioproducts,2023 Apr
3,GSE213570,24,37013645,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,"'Cellulase', 'Fungal growth', 'Gene regulation...",Biotechnology for biofuels and bioproducts,2023 Apr
4,GSE221464,24,36966330,The arabinose transporter MtLat-1 is involved ...,Filamentous fungi possess an array of secrete...,"'Hemicellulase', 'L-Arabinose transport', 'MtA...",Biotechnology for biofuels and bioproducts,2023 Mar
5,GSE214002,8,36691040,The Weimberg pathway: an alternative for Mycel...,With D-xylose being the second most abundant ...,"'1,2,4-Butanetriol', 'D-Xylose', 'D-Xylose deh...",Biotechnology for biofuels and bioproducts,2023 Jan
6,GSE214142,18,36478865,PFK2/FBPase-2 is a potential target for metabo...,The key enzyme 6-phosphofructo-2-kinase (PFK2)...,"'6-phosphofructo-2-kinase/fructose-2,6-biphosp...",Frontiers in microbiology,2022
7,GSE205182,18,36165620,"MtTRC-1, a Novel Transcription Factor, Regulat...",The thermophilic fungus Myceliophthora thermop...,"'ER stress', 'MtHAC-1', 'MtTRC-1', 'Mtcbh-1', ...",Applied and environmental microbiology,2022 Oct
8,GSE184074,12,35257374,Reconstruction and analysis of genome-scale me...,"Myceliophthora thermophila, a thermophilic fun...","'Myceliophthora thermophila', 'genome-scale me...",Biotechnology and bioengineering,2022 Jul
9,GSE183387,49,,,,,,


In [8]:
# df4与df3合并，依据GEO Series列, 以df4为主
df5 = pd.merge(df4, df3, on='GSE Series', how='left')


# df5列顺序调整
df5 = df5[['GSE Series', 
           'Samples', 
           'Title', 
           'Abstract',
           'Country',
           'Institute', 
           'Journal',
           'Year',
           'PMID' 
           ]]

# GSE Series列更名为Series Accession
df5 = df5.rename(columns={'GSE Series': 'Series Accession'})

df5

,Series Accession,Samples,Title,Abstract,Country,Institute,Journal,Year,PMID
0,GSE213576,36,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,NaN,NaN,Biotechnology for biofuels and bioproducts,2023 Apr,37013645
1,GSE213575,4,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,NaN,NaN,Biotechnology for biofuels and bioproducts,2023 Apr,37013645
2,GSE213573,8,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,NaN,NaN,Biotechnology for biofuels and bioproducts,2023 Apr,37013645
3,GSE213570,24,The putative methyltransferase LaeA regulates ...,Filamentous fungi with the ability to use com...,China,"Tianjin institute of industrial biotechnology,...",Biotechnology for biofuels and bioproducts,2023 Apr,37013645
4,GSE221464,24,The arabinose transporter MtLat-1 is involved ...,Filamentous fungi possess an array of secrete...,China,"Tianjin institute of industrial biotechnology,...",Biotechnology for biofuels and bioproducts,2023 Mar,36966330
5,GSE214002,8,The Weimberg pathway: an alternative for Mycel...,With D-xylose being the second most abundant ...,China,"Tianjin Institute of Industrial Biotechnology,...",Biotechnology for biofuels and bioproducts,2023 Jan,36691040
6,GSE214142,18,PFK2/FBPase-2 is a potential target for metabo...,The key enzyme 6-phosphofructo-2-kinase (PFK2)...,China,Chinese Academy of Sciences,Frontiers in microbiology,2022,36478865
7,GSE205182,18,"MtTRC-1, a Novel Transcription Factor, Regulat...",The thermophilic fungus Myceliophthora thermop...,China,TIB CAS,Applied and environmental microbiology,2022 Oct,36165620
8,GSE184074,12,Reconstruction and analysis of genome-scale me...,"Myceliophthora thermophila, a thermophilic fun...",China,"Tianjin Institute of Industrial Biotechnology,...",Biotechnology and bioengineering,2022 Jul,35257374
9,GSE183387,49,,,Germany,Technische Universität Berlin,,,


In [9]:
# df5保存为dataset.xlsx
df5.to_excel('Mt_dataset.xlsx', index=False)